# 💻 Configuração e execução do Pipeline

###Instalar e importar bibliotecas

In [ ]:
# 1. Instala as bibliotecas necessárias (incluindo python-dotenv)
!pip install kagglehub pandas google-cloud-bigquery dbt-bigquery python-dotenv -q
print("✅ Bibliotecas prontas para uso!")

In [ ]:
import os
import pandas as pd
import kagglehub
from google.cloud import bigquery
from dotenv import load_dotenv

###Infraestrutura e Conexão
Configuração do ambiente, autenticação no GCP e ingestão dos dados brutos.


In [ ]:
# Carregar o arquivo .env
load_dotenv()

# Configurações Iniciais
PROJECT_ID = 'crm-analytics-501214'
DATASET_ID = 'ecom_raw'
TABLE_ID = 'raw_auto_sales'
FULL_TABLE_PATH = f"{PROJECT_ID}.{DATASET_ID}.{TABLE_ID}"

###Extração e Carregamento dos dados no Data Warehouse

In [ ]:
print("📥 Iniciando download do Kaggle...")
# 1. Baixa o dataset
path = kagglehub.dataset_download("ddosad/auto-sales-data")
csv_path = f"{path}/Auto Sales data.csv"

# 2. Carrega os dados no Pandas
df = pd.read_csv(csv_path)
print(f"📊 Dados carregados no Pandas. Linhas: {df.shape[0]}, Colunas: {df.shape[1]}")

# 3. Inicializa o cliente do BigQuery
# O método original identifica automaticamente a variável GOOGLE_APPLICATION_CREDENTIALS injetada pelo load_dotenv()
client = bigquery.Client(project=PROJECT_ID)

# 4. Envia os dados para o BigQuery Sandbox (Substitui a tabela se ela já existir)
job_config = bigquery.LoadJobConfig(write_disposition="WRITE_TRUNCATE")

print(f"🚀 Enviando dados para o BigQuery ({FULL_TABLE_PATH})...")
job = client.load_table_from_dataframe(df, FULL_TABLE_PATH, job_config=job_config)
job.result()

print(f"✨ [SUCCESS] Dados brutos carregados com sucesso no BigQuery Sandbox!")

###Execução e teste do dbt


In [ ]:
# Executa as transformações do dbt
%cd CRM_Analytics_Project
!dbt run

# Roda os testes de consistência do dbt
!dbt test
